In [0]:
%sql
use catalog capstone_team1;
use schema bronze;

In [0]:
from pyspark.sql.functions import (
    abs, array, col, count, count_distinct, desc, desc_nulls_last, explode,
    lit, min, round, struct, try_to_timestamp, unix_timestamp,
)

path = "/Volumes/capstone_team1/bronze/raw_data/adls/primary_events/"
columns = [
    "transaction_id", "event_timestamp", "ingest_timestamp", "merchant_id", "customer_id",
    "payment_channel", "payment_status", "currency", "amount", "risk_score",
    "device_id", "source_system", "source_schema_version",
]

df = spark.read.csv(path, header=True, schema=", ".join(f"{c} STRING" for c in columns))
total = df.count()

df.printSchema()
print(f"Rows: {total:,}")

In [0]:
blanks = "'', 'na', 'n/a', 'null', 'none', 'nan'"

stats = df.selectExpr(
    *[f"count_if({c} IS NULL) AS {c}_null" for c in df.columns],
    *[f"count_if({c} IS NULL OR lower(trim({c})) IN ({blanks})) AS {c}_missing" for c in df.columns],
).first()

display(
    spark.createDataFrame(
        [(c, stats[f"{c}_null"], stats[f"{c}_missing"]) for c in df.columns],
        "column string, null_count long, missing_count long",
    )
    .withColumn("missing_pct", round(col("missing_count") * 100 / total, 2))
    .orderBy(desc("missing_count"), "column")
)

In [0]:
by_txn = df.groupBy("transaction_id").agg(
    count("*").alias("occurrences"),
    count_distinct(struct(*[c for c in df.columns if c != "transaction_id"])).alias("versions"),
)

display(
    by_txn.selectExpr(
        "sum(occurrences) AS total_rows",
        "sum(versions) AS distinct_rows",
        "count(*) AS unique_ids",
        "count_if(occurrences > 1) AS repeated_ids",
        "count_if(versions > 1) AS conflicting_ids",
    ).selectExpr("*", "total_rows - distinct_rows AS exact_duplicates", "total_rows - unique_ids AS rows_with_repeated_id")
)

display(by_txn.filter("occurrences > 1").orderBy(desc("occurrences"), desc("versions"), "transaction_id").limit(20))

In [0]:
cat_cols = ["payment_channel", "payment_status", "currency", "source_system", "source_schema_version"]
id_cols = ["transaction_id", "merchant_id", "customer_id", "device_id"]

display(df.agg(*[count_distinct(c).alias(c) for c in cat_cols + id_cols]))

display(
    df.select(explode(array(*[struct(lit(c).alias("column"), col(c).alias("value")) for c in cat_cols])).alias("kv"))
    .groupBy("kv.column", "kv.value")
    .count()
    .withColumn("pct", round(col("count") * 100 / total, 2))
    .orderBy("column", desc("count"))
)

In [0]:
num = df.selectExpr(
    "amount AS amount_raw",
    "risk_score AS risk_raw",
    "try_cast(amount AS DOUBLE) AS amount",
    "try_cast(risk_score AS DOUBLE) AS risk_score",
)

display(num.selectExpr(
    "count_if(amount IS NULL AND amount_raw IS NOT NULL) AS amount_non_numeric",
    "count_if(risk_score IS NULL AND risk_raw IS NOT NULL) AS risk_score_non_numeric",
    "count_if(amount <= 0) AS amount_zero_or_negative",
    "count_if(NOT risk_score BETWEEN 0 AND 100) AS risk_score_out_of_0_100",
))

display(num.select("amount", "risk_score").summary("count", "mean", "stddev", "min", "1%", "25%", "50%", "75%", "99%", "max"))

In [0]:
fmt = lit("yyyy-MM-dd'T'HH:mm:ss")
late = 30

ts = df.select(
    "transaction_id", "event_timestamp", "ingest_timestamp",
    try_to_timestamp(col("event_timestamp"), fmt).alias("event_ts"),
    try_to_timestamp(col("ingest_timestamp"), fmt).alias("ingest_ts"),
).withColumn("latency_min", (unix_timestamp("ingest_ts") - unix_timestamp("event_ts")) / 60)

display(ts.selectExpr(
    "count_if(event_ts IS NULL AND event_timestamp IS NOT NULL) AS event_ts_unparseable",
    "count_if(ingest_ts IS NULL AND ingest_timestamp IS NOT NULL) AS ingest_ts_unparseable",
    "count_if(latency_min < 0) AS ingest_before_event",
    "count_if(latency_min = 0) AS ingest_equals_event",
    "count_if(event_ts > current_timestamp()) AS event_in_future",
    f"count_if(latency_min > {late}) AS late_arrivals_gt_{late}min",
    "min(event_ts) AS min_event_ts", "max(event_ts) AS max_event_ts",
    "min(ingest_ts) AS min_ingest_ts", "max(ingest_ts) AS max_ingest_ts",
))

display(ts.select("latency_min").summary("count", "mean", "min", "25%", "50%", "75%", "95%", "99%", "max"))

display(
    ts.selectExpr("latency_min", """
        CASE WHEN latency_min < 0 THEN 'negative (ingest before event)'
             WHEN latency_min <= 15 THEN '0-15 min'
             WHEN latency_min <= 30 THEN '15-30 min'
             WHEN latency_min <= 60 THEN '30-60 min'
             WHEN latency_min <= 1440 THEN '1h-1 day'
             ELSE '> 1 day' END AS latency_bucket""")
    .groupBy("latency_bucket")
    .agg(count("*").alias("count"), min("latency_min").alias("start"))
    .orderBy("start")
    .drop("start")
)

display(ts.filter(f"latency_min > {late} OR latency_min < 0").orderBy(desc(abs(col("latency_min")))).limit(20))

In [0]:
num_cols = ["amount", "risk_score"]
nums = df.selectExpr(*[f"try_cast({c} AS DOUBLE) AS {c}" for c in num_cols])

fences = []
for c, (q1, q3) in zip(num_cols, nums.approxQuantile(num_cols, [0.25, 0.75], 0.001)):
    iqr = q3 - q1
    fences.append((c, q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr, q1 - 3 * iqr, q3 + 3 * iqr))

r = nums.selectExpr(*[
    expr
    for c, _, _, _, lo, hi, ext_lo, ext_hi in fences
    for expr in (
        f"count_if({c} > {hi}) AS {c}_high",
        f"count_if({c} < {lo}) AS {c}_low",
        f"count_if({c} < {ext_lo} OR {c} > {ext_hi}) AS {c}_extreme",
        f"min({c}) AS {c}_min",
        f"max({c}) AS {c}_max",
    )
]).first()

display(spark.createDataFrame(
    [
        (c, q1, q3, iqr, lo, hi, r[f"{c}_high"] + r[f"{c}_low"], r[f"{c}_high"], r[f"{c}_low"],
         r[f"{c}_extreme"], r[f"{c}_min"], r[f"{c}_max"])
        for c, q1, q3, iqr, lo, hi, _, _ in fences
    ],
    "column string, q1 double, q3 double, iqr double, lower_fence double, upper_fence double, "
    "outliers_1_5_iqr long, high_outliers long, low_outliers long, extreme_3_iqr long, min double, max double",
))

display(df.withColumn("amount_value", col("amount").try_cast("double")).orderBy(desc_nulls_last("amount_value")).limit(10))

## Data profile findings: primary_events (11,540 rows)

| Check | Result | What it means / action for silver |
| --- | --- | --- |
| Nulls | `source_schema_version` 9,000 (78%), `payment_channel` 12 (0.1%); all others 0 | Older files have no schema version, so default it to `v1`. Fill or quarantine the null channels. |
| Missing (blank or placeholder) | Same as nulls; no empty strings or `NA`/`null` tokens | No extra cleaning needed beyond the nulls. |
| Duplicates | 24 exact duplicate rows (`.distinct()` gives 11,516); 40 repeated `transaction_id` (11,500 unique), about 16 have conflicting payloads | Drop exact duplicates. For conflicting ones, keep the latest by `ingest_timestamp`. |
| Categorical | channel: ECOM/ATM/POS; status: FAILED/COMPLETED/PENDING/OPEN (balanced); `currency`=USD, `source_system`=ADLS_EVENT (constant) | The domains are clean. Constant columns add no analytical value. |
| amount | min 1.47, median 22.57, mean 34.97, std 133.2, p99 122.54, max 6,889.50 | Heavily right-skewed: a few very large values inflate the mean and std. |
| risk_score | 1.01 to 25.0, mean 13.03, uniform-like | Within range, no anomalies. |
| Timestamps | 100% parse with `yyyy-MM-dd'T'HH:mm:ss`; event range 2026-07-01 to 2026-08-01; no ingest-before-event, no future dates | Safe to cast to timestamp. Strings carry no timezone, so treat them as UTC. |
| Late arrival | Latency from 1 to 25 min (median 13); 0 records over the 30-min threshold | No late arrivals at a 30-min SLA. Lower `LATE_THRESHOLD_MIN` if your SLA is stricter. |
| Outliers (IQR) | amount: 692 above upper fence 69.40, 218 extreme (>3×IQR), top values are 6,889.50 / 5,430.75 / 4,149, clustered in TRANSACTION-00003xx on 2026-07-01; risk_score: 0 | Flag high amounts (e.g. `is_amount_outlier`) instead of dropping them. They could be real high-value or fraudulent transactions. |